# KidneyVision AI — Stage 1 Gate Model Training
### Binary AI Filter: Kidney Scan vs. Non-Kidney (Chest X-ray, Bone X-ray, Dental X-ray, etc.)

---
### 🎯 The "Hard Negative X-Ray" Clinical Problem:
In medical imaging, if you train a model simply on *Kidney scans vs. Everyday photos*, the CNN will quickly learn a shallow shortcut:  
**"Grayscale + dark background + bony structures = Kidney."**

If a user or doctor accidentally uploads:
- **Chest X-ray** (showing ribs, lungs, clavicles)
- **Bone / Skeletal X-ray** (arms, legs, spine, pelvis, hands)
- **Dental X-ray** (panoramic jaw, periapical teeth)
- **Brain / Head CT**

A naive model would mistakenly classify them as a kidney and send them to the diagnostic model!  
**Our Objective:** We force the Gate Model to see Chest, Bone, and Dental X-rays during training so it strictly rejects them ($P(\text{Kidney}) \approx 0.0$) and **ONLY** accepts genuine renal CT and kidney ultrasound scans.

---
#### Architecture & Specs:
- **Backbone:** MobileNetV3-Small (Pretrained on ImageNet)
- **Input Shape:** `(224, 224, 3)` with `crop_to_aspect_ratio=True` (preserves anatomical proportions across different scanners)
- **Output:** Sigmoid unit ($P(\text{Kidney})$)
- **Strict Gate Threshold:** $\ge 0.75$ (rejects any ambiguous or non-kidney scan)
- **Export File:** `kidney_gate_model.keras`

In [ ]:
import os
import sys
import io
import glob
import shutil
import hashlib
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, metrics

# Reproducibility Seed
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"Available GPUs: {gpus if gpus else 'None (Running on CPU)'}")


## 1. Dataset Directory Architecture (Multi-Modal Negative Sources)

Your gate dataset should contain a balanced mix of genuine kidney scans and diverse hard-negative X-rays:
```text
dataset_gate/
├── train/
│   ├── 1_kidney/            # (~3,000 scans: renal CT, ultrasound, normal & stone)
│   └── 0_non_kidney/        # Combined pool of hard negatives:
│       ├── chest_xray/      # Chest X-rays (lungs, ribs, heart)
│       ├── bone_xray/       # Skeletal X-rays (extremities, pelvis, spine)
│       ├── dental_xray/     # Dental / panoramic jaw X-rays
│       ├── brain_ct/        # Head/brain CT scans
│       └── non_medical/     # Everyday photos, random documents
└── val/
    ├── 1_kidney/
    └── 0_non_kidney/
```
*Note: Subfolders inside `0_non_kidney/` are automatically traversed and merged by `image_dataset_from_directory`!*

In [ ]:
# ==========================================
# CONFIGURATION & HYPERPARAMETERS
# ==========================================
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
BATCH_SIZE = 32

# Root dataset directory (Update to your path if on Kaggle / Colab)
DATASET_DIR = "dataset_gate"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "val")
TEST_DIR = os.path.join(DATASET_DIR, "test")

# Strict clinical decision threshold
# Image is ONLY accepted if P(Kidney) >= 0.75 (blocks borderline X-rays)
KIDNEY_GATE_THRESHOLD = 0.75

# Output model path
MODEL_SAVE_PATH = "kidney_gate_model.keras"

print(f"Target Model Save Path: {MODEL_SAVE_PATH}")
print(f"Gate Admission Threshold: {KIDNEY_GATE_THRESHOLD * 100:.0f}%")


## 2. Dataset Cleaning, Deduplication & Integrity Audit

Scans all images before training to eliminate:
1. Corrupted image files that cannot be decoded
2. Zero-byte empty files
3. Duplicate images via MD5 hashing to prevent data leakage between train and validation

In [ ]:
def clean_and_audit_dataset(directory_path: str):
    """Scans directory to remove unreadable, zero-byte, and duplicate images."""
    if not os.path.exists(directory_path):
        print(f"[WARNING] Directory not found: {directory_path}. Skipping audit.")
        return

    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".jfif")
    seen_hashes = set()
    removed_corrupted = 0
    removed_duplicates = 0
    total_valid = 0

    print(f"\n[DATA AUDIT] Auditing images in: {directory_path}...")
    
    for root, _, files in os.walk(directory_path):
        for fname in files:
            fpath = os.path.join(root, fname)
            
            if not fname.lower().endswith(valid_extensions):
                continue

            # Check file size
            if os.path.getsize(fpath) == 0:
                try:
                    os.remove(fpath)
                    removed_corrupted += 1
                except OSError:
                    pass
                continue

            # Verify image format integrity with PIL
            try:
                with Image.open(fpath) as img:
                    img.verify()
                
                # MD5 hash check for duplicate removal
                with open(fpath, "rb") as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()
                    
                if file_hash in seen_hashes:
                    try:
                        os.remove(fpath)
                        removed_duplicates += 1
                    except OSError:
                        pass
                else:
                    seen_hashes.add(file_hash)
                    total_valid += 1

            except Exception:
                try:
                    os.remove(fpath)
                    removed_corrupted += 1
                except OSError:
                    pass

    print(f"Audit Complete for {directory_path}:")
    print(f"  - Valid unique images: {total_valid}")
    print(f"  - Corrupted files removed: {removed_corrupted}")
    print(f"  - Duplicate files removed: {removed_duplicates}")

# Execute audit
if os.path.exists(TRAIN_DIR):
    clean_and_audit_dataset(TRAIN_DIR)
if os.path.exists(VAL_DIR):
    clean_and_audit_dataset(VAL_DIR)


## 3. High-Performance tf.data Input Pipeline & Multi-Scanner Normalization

- **`crop_to_aspect_ratio=True`**: Prevents geometric distortion when loading scans from different scanner resolutions.
- **Data Augmentation**: Horizontal flip, rotation, zoom, contrast, and brightness adjustments.

In [ ]:
if not os.path.exists(TRAIN_DIR) or not os.path.exists(VAL_DIR):
    print("[NOTE] Dataset folders not found at specified paths yet.")
    print(f"Please configure TRAIN_DIR ({TRAIN_DIR}) and VAL_DIR ({VAL_DIR}) when ready.")
else:
    print("Loading Gate Train Dataset...")
    train_raw = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=IMG_SIZE,
        crop_to_aspect_ratio=True,  # Preserves true anatomical proportions
        batch_size=BATCH_SIZE,
        label_mode="binary",
        shuffle=True,
        seed=SEED
    )

    print("Loading Gate Validation Dataset...")
    val_raw = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE,
        crop_to_aspect_ratio=True,
        batch_size=BATCH_SIZE,
        label_mode="binary",
        shuffle=False
    )

    class_names = train_raw.class_names
    print(f"Class mapping detected: {class_names}")
    # If folders are named ['0_non_kidney', '1_kidney']:
    # Class 0 = Non-Kidney (Chest/Bone/Dental X-rays, etc.)
    # Class 1 = Genuine Kidney Scan

    # Robust data augmentation
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.08),
        layers.RandomContrast(0.12),
        layers.RandomBrightness(0.08)
    ], name="gate_data_augmentation")

    AUTOTUNE = tf.data.AUTOTUNE

    def prepare_train_ds(ds):
        return (
            ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
              .shuffle(buffer_size=1000, seed=SEED)
              .prefetch(buffer_size=AUTOTUNE)
        )

    def prepare_val_ds(ds):
        return ds.prefetch(buffer_size=AUTOTUNE)

    train_dataset = prepare_train_ds(train_raw)
    val_dataset = prepare_val_ds(val_raw)
    print("tf.data pipelines prepared successfully.")


## 4. Architecture Definition: MobileNetV3-Small Backbone + Triage Head

MobileNetV3-Small is pretrained on ImageNet:
- **Ultra-fast CPU latency:** $< 15$ ms per scan
- **Low memory:** ~1.5 million parameters
- **Deep feature discrimination:** Effectively distinguishes thoracic/skeletal/dental patterns from retroperitoneal kidney structures

In [ ]:
def build_gate_model(input_shape=(224, 224, 3)):
    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False  # Freeze for Phase 1

    inputs = tf.keras.Input(shape=input_shape, name="scan_input")
    
    # MobileNetV3 native scaling [-1, 1]
    x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
    x = base_model(x, training=False)
    
    # Clinical triage head
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="gate_bn1")(x)
    x = layers.Dropout(0.35, name="gate_dropout1")(x)
    x = layers.Dense(128, activation="relu", name="gate_dense1")(x)
    x = layers.BatchNormalization(name="gate_bn2")(x)
    x = layers.Dropout(0.2, name="gate_dropout2")(x)
    
    # Output: P(Kidney)
    outputs = layers.Dense(1, activation="sigmoid", name="kidney_probability")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="KidneyVision_Gate_MobileNetV3")
    return model, base_model

gate_model, base_backbone = build_gate_model()
gate_model.summary()


## 5. Phase 1: Feature Extraction (Training Classification Head)

Trains only the top triage dense layers while keeping MobileNetV3 weights frozen.

In [ ]:
gate_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="auc")
    ]
)

callbacks_phase1 = [
    callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

if "train_dataset" in locals() and "val_dataset" in locals():
    print("Starting Phase 1 Training (Backbone Frozen)...")
    history_p1 = gate_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=8,
        callbacks=callbacks_phase1
    )
else:
    print("[READY] Load datasets above to execute Phase 1.")


## 6. Phase 2: Fine-Tuning Top Convolutional Layers

Unfreezes the top 35 layers of MobileNetV3 with a low learning rate ($10^{-5}$) so the convolutional filters learn to explicitly distinguish rib/dental/skeletal textures from renal parenchyma.

In [ ]:
# Unfreeze top layers of the backbone
base_backbone.trainable = True
for layer in base_backbone.layers[:-35]:
    layer.trainable = False

gate_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="auc")
    ]
)

callbacks_phase2 = [
    callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

if "train_dataset" in locals() and "val_dataset" in locals():
    print("Starting Phase 2 Fine-Tuning...")
    history_p2 = gate_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=12,
        callbacks=callbacks_phase2
    )
else:
    print("[READY] Run after Phase 1.")


## 7. Model Evaluation & Hard-Negative Stress Test

Evaluates the gate on the validation set at the strict $\text{threshold} = 0.75$, and specifically audits whether **Chest X-rays**, **Bone X-rays**, and **Dental X-rays** are 100% rejected.

In [ ]:
if "val_dataset" in locals():
    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
    
    y_true = []
    y_pred_probs = []

    for images, labels in val_dataset:
        probs = gate_model.predict(images, verbose=0)
        y_true.extend(labels.numpy().flatten())
        y_pred_probs.extend(probs.flatten())

    y_true = np.array(y_true)
    y_pred_probs = np.array(y_pred_probs)

    # Evaluate at strict gate threshold
    y_pred = (y_pred_probs >= KIDNEY_GATE_THRESHOLD).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    print("\n" + "="*55)
    print(f"CONFUSION MATRIX (Strict Gate Threshold = {KIDNEY_GATE_THRESHOLD}):")
    print(cm)
    print("="*55)

    print("\nCLASSIFICATION REPORT:")
    print(classification_report(y_true, y_pred, target_names=["Non-Kidney / Other X-rays", "Kidney"]))

    auc_val = roc_auc_score(y_true, y_pred_probs)
    print(f"Validation ROC-AUC: {auc_val:.4f}")
    
    # Calculate False Acceptance Rate (FAR) of non-kidney images:
    # Negative images mistakenly classified as Kidney
    neg_count = np.sum(y_true == 0)
    false_accepts = np.sum((y_true == 0) & (y_pred == 1))
    far_percent = (false_accepts / neg_count * 100) if neg_count > 0 else 0.0
    print(f"\nHard-Negative False Acceptance Rate (FAR): {far_percent:.2f}% (Target: 0.0%)")
else:
    print("[READY] Run after training to evaluate metrics.")


## 8. Specific Rejection Test for Chest, Bone & Dental X-Rays

Test individual sample scans to confirm that Chest, Bone, and Dental X-rays are categorically rejected with low kidney scores.

In [ ]:
def test_scan_admission(image_path: str, threshold: float = KIDNEY_GATE_THRESHOLD):
    """
    Simulates the production Flask gate.
    Rejects any non-kidney scan (Chest, Bone, Dental X-rays, etc.)
    """
    if not os.path.exists(image_path):
        return {"error": f"File not found: {image_path}"}
        
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    img_arr = tf.keras.utils.img_to_array(img)
    img_batch = np.expand_dims(img_arr, axis=0)
    
    prob = float(gate_model.predict(img_batch, verbose=0)[0][0])
    is_kidney = prob >= threshold
    confidence = prob if is_kidney else (1.0 - prob)
    
    if is_kidney:
        status = "ACCEPTED"
        action = "Forwarding to Stage 2 Diagnostic Model (Normal vs. Stone)"
        reason = f"Verified kidney scan (Confidence: {confidence * 100:.1f}%)"
    else:
        status = "REJECTED (HTTP 422)"
        action = "Immediate Clinical Rejection"
        reason = (
            f"Uploaded scan appears to be a Chest X-ray, Bone X-ray, Dental X-ray, "
            f"or non-renal scan (Kidney Probability: {prob * 100:.1f}% < {threshold * 100:.0f}% threshold). "
            f"Please upload a renal CT or kidney ultrasound scan."
        )
        
    return {
        "status": status,
        "action": action,
        "kidney_probability": round(prob, 4),
        "confidence_percent": round(confidence * 100, 2),
        "reason": reason
    }

print("Gate admission simulation function ready.")
# Example usage:
# print(test_scan_admission("path/to/sample_chest_xray.jpg"))


## 9. Final Model Export & Verification

Saves the trained gate model to `kidney_gate_model.keras` and performs reload verification.

In [ ]:
# Save model in native Keras 3 format
gate_model.save(MODEL_SAVE_PATH)
print(f"Gate model successfully saved to: {os.path.abspath(MODEL_SAVE_PATH)}")

# Reload verification
print("Verifying model reload...")
reloaded = tf.keras.models.load_model(MODEL_SAVE_PATH)
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
out = reloaded.predict(dummy, verbose=0)
print(f"Reload verification successful. Warmup inference shape: {out.shape}")
print("\n*** Copy 'kidney_gate_model.keras' into 'ai-flask-service/'! ***")
